# Music Annotation Project — Notebook 39
## V8.2j Reconstruction-Sensitivity by Musical Structure

**Runtime:** CPU. Run all until the interactive semantic-review cell, complete that review, stop the app, and then run the remaining cells.

### Question
Which musical structures benefit most when observed print is protected before reconstruction and can act as context for filling the remaining removal mask?

### Counterfactuals
1. **Frozen baseline:** exact cached V7.8 `geometry_t80`.
2. **Direct preservation:** baseline plus exact selected observed bytes.
3. **Seeded reconstruction:** selected observed bytes are excluded from the reconstruction mask; Telea and `geometry_t80` are rerun on the residual mask.

The seeded output is an experimental development counterfactual—not frozen V7.8 and not a promotion candidate. The primary metric is local `baseline MAE − seeded MAE`; positive values favour seeded reconstruction. Direct-preservation gain and reconstruction spillover beyond copied pixels are reported separately, alongside tolerant ink F1, edge F1 and clean-skeleton recall.

Fine semantic categories cannot be recovered reliably from horizontal/vertical/compact route maps. Staff lines, stems/bar lines, beams, noteheads, slurs/ties and hairpins therefore require human labels; morphology supplies suggestions only. Clean truth is post-hoc scoring/review evidence, never reconstruction input. Inputs are hash-verified, outside-mask pixels are immutable, and incomplete review fails closed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-image gradio openpyxl

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,platform,sys,shutil,zipfile
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
from skimage.morphology import skeletonize
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82C_ROOT=ROOT/'observed_print_classifier_v8_2c_label_sensitivity';V82_ROOT=ROOT/'observed_print_classifier_v8_2';OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2j_reconstruction_sensitivity'
SOURCE_V82C_RUN_OVERRIDE=None;SOURCE_V82_RUN_OVERRIDE=None
THRESHOLDS=[.75,.65,.50,.40];PRIMARY_HIGH=.75;PRIMARY_LOW=.50;MIN_PRINT_RUN=2
GROUP_DILATION_RADIUS=4;LOCAL_EVAL_DILATION=5;PATCH_MARGIN=44;MIN_REGION_PIXELS=2;MAX_REVIEW_REGIONS=300;BOOTSTRAP_RESAMPLES=5000;SEED=42
ALLOW_PARTIAL_RANKING=False
STRUCTURE_TYPES=['STAFF_LINE','STEM_OR_BARLINE','BEAM','NOTEHEAD_OR_COMPACT_GLYPH','SLUR_OR_TIE','CRESCENDO_DECRESCENDO_HAIRPIN','OTHER_PRINTED_SYMBOL','MIXED_PRINTED_STRUCTURES','ANNOTATION_OR_NON_PRINT','INDETERMINATE']
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook39_reconstruction_sensitivity';OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';AUDIT=OUT/'audit';VISUAL=OUT/'visuals';REVIEW=OUT/'review';PAGES=OUT/'threshold_pages';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,VISUAL,REVIEW,PAGES,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Resolve and hash source artefacts

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def image_map(d):
 d=Path(d);return {p.stem:p for p in d.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if d.exists() else {}
def complete_runs(root,req):
 root=Path(root);return sorted([p for p in root.iterdir() if p.is_dir() and all((p/r).exists() for r in req)],key=lambda p:p.stat().st_mtime) if root.exists() else []
def write_json(p,x):Path(p).write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def loadz(p,req):
 with np.load(p) as z:
  miss=[k for k in req if k not in z.files]
  if miss:raise RuntimeError(f'{p}: missing {miss}')
  return {k:z[k].copy() for k in req}
def tn(t):return f'p{int(round(1000*t)):04d}'

hits=complete_runs(V82C_ROOT,['experiment_identity.json','tables/all_eligible_label_sensitivity.parquet']);V82C=Path(SOURCE_V82C_RUN_OVERRIDE) if SOURCE_V82C_RUN_OVERRIDE else (hits[-1] if hits else None)
if V82C is None:raise RuntimeError('No complete Notebook 32 run')
id32=json.loads((V82C/'experiment_identity.json').read_text())
if id32.get('version')!='8.2c-whole-population-label-sensitivity-audit':raise RuntimeError('Unexpected Notebook 32 identity')
V82=Path(SOURCE_V82_RUN_OVERRIDE) if SOURCE_V82_RUN_OVERRIDE else Path(id32['source_v82_run'])
if not V82.exists():
 hits=complete_runs(V82_ROOT,['experiment_identity.json','tables/all_eligible_oof_predictions.parquet','audit/input_and_label_audit.csv']);V82=hits[-1] if hits else None
if V82 is None:raise RuntimeError('No complete V8.2 run')
id82=json.loads((V82/'experiment_identity.json').read_text())
if id82.get('version')!='8.2-development-whole-page' or id82.get('independent_data_used') is not False:raise RuntimeError('Notebook 39 requires development-only V8.2')
enriched=pd.read_parquet(V82C/'tables'/'all_eligible_label_sensitivity.parquet');pred=pd.read_parquet(V82/'tables'/'all_eligible_oof_predictions.parquet')
for d in [enriched,pred]:
 d['page']=d.page.astype(str)
 for c in ['y','x','truth','fold']:d[c]=pd.to_numeric(d[c],errors='raise').astype(int)
 d['p_print']=pd.to_numeric(d.p_print,errors='raise').astype(float)
 if d.duplicated(['page','y','x']).any() or not np.isfinite(d.p_print).all():raise RuntimeError('Invalid prediction rows')
chk=enriched[['page','source_group','y','x','truth','fold','p_print']].merge(pred[['page','source_group','y','x','truth','fold','p_print']],on=['page','source_group','y','x'],suffixes=('_32','_82'),validate='one_to_one')
if len(chk)!=len(pred) or not np.array_equal(chk.truth_32,chk.truth_82) or not np.array_equal(chk.fold_32,chk.fold_82) or not np.allclose(chk.p_print_32,chk.p_print_82,rtol=0,atol=1e-12):raise RuntimeError('Notebook 32/V8.2 lineage mismatch')
V75=Path(id82['v7_5_reference']);status=pd.read_csv(V75/'tables'/'cache_status.csv');status['page']=status.page.astype(str);CACHE={r.page:Path(r.cache) for _,r in status.iterrows()};CLEAN=image_map(ROOT/'source'/'clean');OBS=image_map(ROOT/'aligned');ia=pd.read_csv(V82/'audit'/'input_and_label_audit.csv');ia['page']=ia.page.astype(str);ia=ia.set_index('page')
locks=[]
for page in sorted(enriched.page.unique()):
 if page not in CLEAN or page not in OBS or page not in CACHE or page not in ia.index:raise RuntimeError(f'{page}: missing input')
 exp=[str(ia.loc[page,'clean_sha256']),str(ia.loc[page,'observed_sha256']),str(ia.loc[page,'cache_sha256'])];act=[sha256_file(CLEAN[page]),sha256_file(OBS[page]),sha256_file(CACHE[page])]
 if exp!=act:raise RuntimeError(f'{page}: hash mismatch')
 locks.append({'page':page,'clean':str(CLEAN[page]),'observed':str(OBS[page]),'cache':str(CACHE[page]),'clean_sha256':act[0],'observed_sha256':act[1],'cache_sha256':act[2]})
pd.DataFrame(locks).to_csv(AUDIT/'resolved_hash_verified_inputs.csv',index=False)
SOURCE_FILES=[V82C/'experiment_identity.json',V82C/'tables'/'all_eligible_label_sensitivity.parquet',V82/'experiment_identity.json',V82/'tables'/'all_eligible_oof_predictions.parquet',V82/'audit'/'input_and_label_audit.csv'];write_json(AUDIT/'source_file_hashes.json',{str(p):sha256_file(p) for p in SOURCE_FILES})
print('V8.2:',V82,'pages:',enriched.page.nunique(),'rows:',len(enriched))

## 3. Exact geometry_t80 and counterfactual reconstruction functions

In [ ]:
CROP_MARGIN=48;GEOMETRY_THRESHOLD=80;BAND_WIDTH=4
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0]
 if not np.array_equal(out[mask==0],original[mask==0]):raise RuntimeError('Outside-mask drift')
 return out
def boxes(mask):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;b=[]
 for i in range(1,n):
  x,y,bw,bh,a=stats[i]
  if a>=8:b.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
 changed=True
 while changed:
  changed=False;merged=[]
  while b:
   ax0,ay0,ax1,ay1=b.pop();rest=[]
   for bx0,by0,bx1,by1 in b:
    if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   b=rest;merged.append((ax0,ay0,ax1,ay1))
  b=merged
 return b
def telea(obs,mask):
 out=obs.copy()
 for x0,y0,x1,y1 in boxes(mask):
  mk=np.where(mask[y0:y1,x0:x1]>0,255,0).astype(np.uint8);cand=cv2.inpaint(obs[y0:y1,x0:x1],mk,3,cv2.INPAINT_TELEA);q=out[y0:y1,x0:x1];q[mk>0]=cand[mk>0]
 return immutable(obs,out,mask)
def background(obs,mask):
 v=(mask==0)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*v,(0,0),45);den=cv2.GaussianBlur(v.astype(np.float32),(0,0),45);fb=cv2.GaussianBlur(obs,(0,0),45);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fb),0,255).astype(np.uint8)
def supports(obs,mask):
 dark=(obs<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,st,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=st[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=1
 k=np.ones((9,9),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0
def geometry(obs,mask,tel):
 paper=background(obs,mask);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);h,v,c=supports(obs,mask);m=mask>0;pres=m&(h|v|c)&(obs<200);rem=m&~pres&(obs>=80)&(norm>=80);amb=m&~pres&~rem;out=obs.copy();out[rem]=paper[rem];out[amb]=tel[amb];out[pres]=obs[pres];return immutable(obs,out,mask),{'preserve':pres,'remove':rem,'ambiguous':amb,'horizontal':h,'vertical':v,'compact':c}
def ink(a):return cv2.adaptiveThreshold(a,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge(a):return cv2.Canny(a,70,170)>0
def scores(ref,pred,reg):
 ref=ref&reg;pred=pred&reg
 if not ref.any():return np.nan,np.nan,np.nan
 k=np.ones((3,3),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0;p=(pred&rd).sum()/max(pred.sum(),1);r=(ref&pd).sum()/ref.sum();return p,r,2*p*r/max(p+r,1e-12)
def metrics(clean,cand,reg):
 _,_,if1=scores(ink(clean),ink(cand),reg);_,_,ef1=scores(edge(clean),edge(cand),reg);sk=skeletonize(ink(clean));near=cv2.dilate(ink(cand).astype(np.uint8),np.ones((3,3),np.uint8))>0;return {'mae':float(np.abs(clean.astype(float)-cand.astype(float))[reg].mean()),'ink_f1':if1,'edge_f1':ef1,'skeleton_recall':float((sk&reg&near).sum()/max((sk&reg).sum(),1))}
t=np.full((31,31),255,np.uint8);m=np.zeros_like(t);m[12:19,12:19]=255;assert np.array_equal(telea(t,m)[m==0],t[m==0]);print('Self-test passed')

## 4. Exact baseline replay and threshold pages

In [ ]:
def select(g,shape,t):
 pm=np.zeros(shape,np.uint8);take=g.p_print.to_numpy()>=t;pm[g.y.to_numpy(int)[take],g.x.to_numpy(int)[take]]=1;n,lab,st,_=cv2.connectedComponentsWithStats(pm,8);out=np.zeros(shape,bool)
 for i in range(1,n):
  if st[i,cv2.CC_STAT_AREA]>=MIN_PRINT_RUN:out[lab==i]=1
 return out
REQ=['mask','telea','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact'];runtime={};replay=[];page_rows=[]
for page,g in enriched.groupby('page',sort=False):
 clean=cv2.imread(str(CLEAN[page]),0);obs=cv2.imread(str(OBS[page]),0);a=loadz(CACHE[page],REQ);built,routes=geometry(obs,a['mask'],a['telea']);rex=all(np.array_equal(routes[k],a['g80_'+k]>0) for k in routes)
 replay.append({'page':page,'geometry_exact':np.array_equal(built,a['geometry_t80']),'routes_exact':rex,'outside_immutable':np.array_equal(a['geometry_t80'][a['mask']==0],obs[a['mask']==0])})
 if not all(replay[-1].values()):raise RuntimeError(f'{page}: exact baseline replay failed')
 sels={t:select(g,obs.shape,t) for t in THRESHOLDS};runtime[page]={'clean':clean,'obs':obs,'mask':a['mask']>0,'baseline':a['geometry_t80'],'routes':routes,'rows':g,'sels':sels,'outputs':{}}
 folder=PAGES/page;folder.mkdir();cv2.imwrite(str(folder/'geometry_t80.png'),a['geometry_t80']);cv2.imwrite(str(folder/'clean.png'),clean);cv2.imwrite(str(folder/'observed.png'),obs)
 for t,s in sels.items():
  direct=a['geometry_t80'].copy();direct[s]=obs[s];res=(a['mask']>0)&~s;seeded,sr=geometry(obs,res.astype(np.uint8)*255,telea(obs,res.astype(np.uint8)*255))
  if not np.array_equal(seeded[~(a['mask']>0)],obs[~(a['mask']>0)]) or not np.array_equal(seeded[s],obs[s]):raise RuntimeError(f'{page} {t}: counterfactual integrity failed')
  runtime[page]['outputs'][t]={'direct':direct,'seeded':seeded,'residual':res};cv2.imwrite(str(folder/f'{tn(t)}_direct.png'),direct);cv2.imwrite(str(folder/f'{tn(t)}_seeded.png'),seeded)
  page_rows.append({'page':page,'source_group':g.source_group.iloc[0],'threshold':t,'selected':int(s.sum()),'residual':int(res.sum()),'seeded_change_beyond_direct':int(((seeded!=direct)&~s&(a['mask']>0)).sum()),'direct_sha256':array_hash(direct),'seeded_sha256':array_hash(seeded)})
pd.DataFrame(replay).to_csv(AUDIT/'geometry_t80_replay.csv',index=False);pd.DataFrame(page_rows).to_csv(TABLES/'threshold_page_outputs.csv',index=False);print('Replayed pages:',len(runtime))

## 5. Local reconstruction-sensitivity regions

In [ ]:
rows=[];memory={};kg=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9));ke=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(11,11));ordered=sorted(THRESHOLDS,reverse=True)
for page,z in runtime.items():
 for hi,lo in zip(ordered[:-1],ordered[1:]):
  delta=z['sels'][lo]&~z['sels'][hi];n,lab,st,_=cv2.connectedComponentsWithStats((cv2.dilate(delta.astype(np.uint8),kg)>0).astype(np.uint8),8)
  for cid in range(1,n):
   core=(lab==cid)&delta
   if core.sum()<MIN_REGION_PIXELS:continue
   yy,xx=np.where(core);x0=max(0,xx.min()-PATCH_MARGIN);x1=min(core.shape[1],xx.max()+PATCH_MARGIN+1);y0=max(0,yy.min()-PATCH_MARGIN);y1=min(core.shape[0],yy.max()+PATCH_MARGIN+1);reg=(cv2.dilate(core.astype(np.uint8),ke)>0)&z['mask'];base=z['baseline'];direct=z['outputs'][lo]['direct'];seeded=z['outputs'][lo]['seeded'];mb=metrics(z['clean'],base,reg);md=metrics(z['clean'],direct,reg);ms=metrics(z['clean'],seeded,reg);rid=f'{page}__{tn(hi)}_to_{tn(lo)}__{cid:04d}';hf=z['routes']['horizontal'][core].mean();vf=z['routes']['vertical'][core].mean();cf=z['routes']['compact'][core].mean();suggest='STAFF_LINE' if hf>=.5 and vf<.3 else ('STEM_OR_BARLINE' if vf>=.5 and hf<.3 else ('NOTEHEAD_OR_COMPACT_GLYPH' if cf>=.5 else 'INDETERMINATE'));r={'region_id':rid,'page':page,'source_group':z['rows'].source_group.iloc[0],'higher_threshold':hi,'lower_threshold':lo,'selected_pixels':int(core.sum()),'eval_pixels':int(reg.sum()),'x0':x0,'y0':y0,'x1':x1,'y1':y1,'heuristic_suggestion':suggest}
   for name,m in [('baseline',mb),('direct',md),('seeded',ms)]:
    for k,v in m.items():r[name+'_'+k]=v
   for k in mb:r['direct_'+k+'_gain']=mb[k]-md[k] if k=='mae' else md[k]-mb[k];r['seeded_'+k+'_gain']=mb[k]-ms[k] if k=='mae' else ms[k]-mb[k];r['spillover_'+k+'_gain']=md[k]-ms[k] if k=='mae' else ms[k]-md[k]
   rows.append(r);memory[rid]={'page':page,'core':core,'box':(x0,y0,x1,y1),'lo':lo}
regions=pd.DataFrame(rows).sort_values(['seeded_mae_gain','selected_pixels'],ascending=False).head(MAX_REVIEW_REGIONS).reset_index(drop=True)
if regions.empty:raise RuntimeError('No regions')
regions.to_csv(TABLES/'candidate_region_metrics.csv',index=False);print('Regions:',len(regions))

## 6. Review panels and semantic queue

In [ ]:
def panel(im,title):
 q=cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else im.copy();cv2.rectangle(q,(0,0),(q.shape[1],26),(255,255,255),-1);cv2.putText(q,title,(5,18),cv2.FONT_HERSHEY_SIMPLEX,.48,(0,0,0),1);return q
def render(rid):
 m=memory[rid];z=runtime[m['page']];x0,y0,x1,y1=m['box'];core=m['core'][y0:y1,x0:x1];obs=z['obs'][y0:y1,x0:x1];ov=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR);ov[core]=(0,255,255);lo=m['lo'];ims=[panel(ov,'Observed + selected'),panel(z['clean'][y0:y1,x0:x1],'Clean truth'),panel(z['baseline'][y0:y1,x0:x1],'geometry_t80'),panel(z['outputs'][lo]['direct'][y0:y1,x0:x1],'Direct preserve'),panel(z['outputs'][lo]['seeded'][y0:y1,x0:x1],'Seeded reconstruction')];return np.concatenate(ims,axis=1)
queue=[]
for _,r in regions.iterrows():
 p=REVIEW/f'{r.region_id}.png';cv2.imwrite(str(p),render(r.region_id));queue.append({'region_id':r.region_id,'page':r.page,'heuristic_suggestion':r.heuristic_suggestion,'structure_type':'','review_confidence':'','review_note':'','reviewed_utc':'','image_path':str(p)})
REVIEW_CSV=TABLES/'semantic_structure_review.csv';pd.DataFrame(queue).to_csv(REVIEW_CSV,index=False);print(REVIEW_CSV)

## 7. Human semantic review
Label the printed structure being completed. The clean panel is available solely to resolve semantic identity. The route-map suggestion is an aid, not truth. Use `INDETERMINATE` when necessary. Every save is durable.

In [ ]:
import gradio as gr
state=pd.read_csv(REVIEW_CSV).fillna('')
def nxt(start=0):
 q=np.where(state.structure_type.astype(str).str.strip().eq(''))[0];return int(q[0]) if len(q) else min(int(start),len(state)-1)
def load(i):
 i=max(0,min(int(i),len(state)-1));r=state.iloc[i];img=cv2.cvtColor(cv2.imread(r.image_path),cv2.COLOR_BGR2RGB);m=regions.set_index('region_id').loc[r.region_id];info=f'{i+1}/{len(state)} | {r.region_id} | suggestion only: {r.heuristic_suggestion} | seeded MAE gain={m.seeded_mae_gain:.3f} | spillover={m.spillover_mae_gain:.3f}';return i,img,info,r.structure_type or None,r.review_confidence or None,r.review_note
def save(i,st,conf,note):
 global state;i=int(i)
 if st not in STRUCTURE_TYPES or conf not in ['HIGH','MEDIUM','LOW']:return (*load(i),'Choose structure and confidence')
 state.loc[i,['structure_type','review_confidence','review_note','reviewed_utc']]=[st,conf,note or '',datetime.now(timezone.utc).isoformat()];state.to_csv(REVIEW_CSV,index=False);return (*load(nxt(i+1)),'Saved')
def go(i,d):return (*load(int(i)+d),'')
start=nxt()
with gr.Blocks(title='Notebook 39 semantic review') as demo:
 gr.Markdown('# Notebook 39 — structure review');idx=gr.Number(value=start,visible=False);im=gr.Image(type='numpy');info=gr.Textbox(interactive=False);st=gr.Dropdown(STRUCTURE_TYPES,label='Structure');conf=gr.Radio(['HIGH','MEDIUM','LOW']);note=gr.Textbox(label='Note');status=gr.Textbox(interactive=False)
 with gr.Row():prev=gr.Button('Previous');sv=gr.Button('Save and next',variant='primary');ne=gr.Button('Next')
 demo.load(lambda:load(start),outputs=[idx,im,info,st,conf,note]);prev.click(lambda i:go(i,-1),idx,[idx,im,info,st,conf,note,status]);ne.click(lambda i:go(i,1),idx,[idx,im,info,st,conf,note,status]);sv.click(save,[idx,st,conf,note],[idx,im,info,st,conf,note,status])
demo.launch(share=False,debug=True)

## 8. Fail-closed ranking with page-clustered uncertainty

In [ ]:
review=pd.read_csv(REVIEW_CSV).fillna('');missing=review.structure_type.astype(str).str.strip().eq('');gate={'regions':len(review),'reviewed':int((~missing).sum()),'unreviewed':int(missing.sum()),'complete':bool(not missing.any()),'partial_allowed':ALLOW_PARTIAL_RANKING};write_json(AUDIT/'semantic_review_gate.json',gate)
if missing.any() and not ALLOW_PARTIAL_RANKING:raise RuntimeError(f'Review incomplete: {missing.sum()} regions')
a=regions.merge(review[['region_id','structure_type','review_confidence','review_note','reviewed_utc']],on='region_id',validate='one_to_one');a=a[a.structure_type.astype(str).str.strip().ne('')];a.to_csv(TABLES/'reviewed_region_metrics.csv',index=False)
def pci(g,col):
 v=g.groupby('page')[col].mean().dropna().to_numpy(float)
 if not len(v):return np.nan,np.nan,np.nan
 rng=np.random.default_rng(SEED);b=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_RESAMPLES)]);return v.mean(),np.percentile(b,2.5),np.percentile(b,97.5)
rr=[];cols=['direct_mae_gain','seeded_mae_gain','spillover_mae_gain','seeded_ink_f1_gain','seeded_edge_f1_gain','seeded_skeleton_recall_gain']
for st,g in a.groupby('structure_type'):
 r={'structure_type':st,'regions':len(g),'pages':g.page.nunique(),'source_groups':g.source_group.nunique(),'selected_pixels':int(g.selected_pixels.sum())}
 for c in cols:r[c+'_page_mean'],r[c+'_ci95_low'],r[c+'_ci95_high']=pci(g,c)
 r['positive_seeded_regions']=int((g.seeded_mae_gain>0).sum());r['positive_spillover_regions']=int((g.spillover_mae_gain>0).sum());rr.append(r)
ranking=pd.DataFrame(rr).sort_values('seeded_mae_gain_page_mean',ascending=False).reset_index(drop=True);ranking.insert(0,'rank',np.arange(1,len(ranking)+1));ranking.to_csv(TABLES/'structure_reconstruction_sensitivity_ranking.csv',index=False)
pair=a.groupby(['structure_type','higher_threshold','lower_threshold'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),selected_pixels=('selected_pixels','sum'),seeded_mae_gain=('seeded_mae_gain','mean'),spillover_mae_gain=('spillover_mae_gain','mean'),seeded_ink_f1_gain=('seeded_ink_f1_gain','mean'),seeded_edge_f1_gain=('seeded_edge_f1_gain','mean'),seeded_skeleton_recall_gain=('seeded_skeleton_recall_gain','mean'));pair.to_csv(TABLES/'structure_ranking_by_threshold_pair.csv',index=False)
primary=a[(a.higher_threshold<=PRIMARY_HIGH)&(a.lower_threshold>=PRIMARY_LOW)].groupby('structure_type',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),seeded_mae_gain=('seeded_mae_gain','mean'),spillover_mae_gain=('spillover_mae_gain','mean'),seeded_ink_f1_gain=('seeded_ink_f1_gain','mean'),seeded_edge_f1_gain=('seeded_edge_f1_gain','mean'),seeded_skeleton_recall_gain=('seeded_skeleton_recall_gain','mean')).sort_values('seeded_mae_gain',ascending=False);primary.to_csv(TABLES/'primary_075_vs_050_structure_summary.csv',index=False);display(ranking)

## 9. Charts, manifest and final upload package

In [ ]:
q=ranking.sort_values('seeded_mae_gain_page_mean');x=q.seeded_mae_gain_page_mean.to_numpy();lo=x-q.seeded_mae_gain_ci95_low.to_numpy();hi=q.seeded_mae_gain_ci95_high.to_numpy()-x;plt.figure(figsize=(10,max(5,.55*len(q))));plt.barh(q.structure_type,x,xerr=np.vstack([lo,hi]));plt.axvline(0,color='black');plt.xlabel('MAE improvement: geometry_t80 − seeded reconstruction');plt.tight_layout();plt.savefig(VISUAL/'structure_reconstruction_sensitivity_ranking.png',dpi=180);plt.close()
source_now={str(p):sha256_file(p) for p in SOURCE_FILES}
if source_now!=json.loads((AUDIT/'source_file_hashes.json').read_text()):raise RuntimeError('Source changed during run')
best=ranking.iloc[0];summary=['NOTEBOOK 39 — RECONSTRUCTION SENSITIVITY','',f'Pages: {len(runtime)}',f'Reviewed regions: {len(a)}',f'Highest descriptive structure: {best.structure_type}',f'Page-mean seeded MAE gain: {best.seeded_mae_gain_page_mean:.6f}',f'95% page-cluster bootstrap interval: [{best.seeded_mae_gain_ci95_low:.6f}, {best.seeded_mae_gain_ci95_high:.6f}]','','Positive spillover means preserved observations improved reconstruction beyond copied pixels.','Development-only; no threshold or production authority.'];(OUT/'executive_summary.txt').write_text(chr(10).join(summary));print(chr(10).join(summary))
manifest={'version':'39-v8.2j','created_utc':datetime.now(timezone.utc).isoformat(),'development_only':True,'independent_data_used':False,'source_v82':str(V82),'source_v82c':str(V82C),'v75':str(V75),'thresholds':THRESHOLDS,'primary':[PRIMARY_HIGH,PRIMARY_LOW],'semantic_review':gate,'source_hashes':source_now,'primary_metric':'page-clustered local baseline MAE minus seeded MAE'};write_json(OUT/'experiment_manifest.json',manifest)
arts=[OUT/'executive_summary.txt',OUT/'experiment_manifest.json',TABLES/'structure_reconstruction_sensitivity_ranking.csv',TABLES/'primary_075_vs_050_structure_summary.csv',TABLES/'structure_ranking_by_threshold_pair.csv',TABLES/'reviewed_region_metrics.csv',TABLES/'semantic_structure_review.csv',TABLES/'threshold_page_outputs.csv',AUDIT/'geometry_t80_replay.csv',AUDIT/'resolved_hash_verified_inputs.csv',AUDIT/'source_file_hashes.json',AUDIT/'semantic_review_gate.json',VISUAL/'structure_reconstruction_sensitivity_ranking.png']
miss=[str(p) for p in arts if not p.exists()]
if miss:raise RuntimeError(f'Missing upload artefacts: {miss}')
for p in arts:
 d=UPLOAD/p.relative_to(OUT);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
(UPLOAD/'README.txt').write_text('Notebook 39 development-only reconstruction-sensitivity audit. Positive MAE gain favours seeded reconstruction. Spillover is improvement beyond direct preserved pixels.'+chr(10));inv=[]
for p in sorted(UPLOAD.rglob('*')):
 if p.is_file():inv.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False);bundle=OUT/'notebook39_reconstruction_sensitivity_upload_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,p.relative_to(UPLOAD))
print('Bundle:',bundle)